# IMS OL-vs-DA Tables and Maps (from precomputed script outputs)

This notebook is lightweight by design: it reads outputs written by `run_ims_ol_da_cell_metrics.py` and focuses on quick analysis and plotting.

Outputs expected:
- `ims_ol_da_comparison_table_*.parquet/csv`
- `ims_ol_da_pair_daily_*.parquet/csv`
- `ims_ol_da_scope_metadata_*.csv`
- `ims_ol_da_cell_counts_metrics_*.nc4`


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

SEASON_ORDER = ["DJF", "MAM", "JJA", "SON"]

DOMAIN = "SMAP_EASEv2_M36_GLOBAL"
YEAR_START = 2000
YEAR_END = 2024
SCF_THRESHOLD = 0.5

OUTPUT_DIR = Path("/discover/nobackup/projects/land_da/geosldas-analysis/projects/IMS/output")
CACHE_TAG = f"{DOMAIN}_{YEAR_START}_{YEAR_END}_thr{SCF_THRESHOLD:.2f}".replace(".", "p")

COMPARISON_PARQUET = OUTPUT_DIR / f"ims_ol_da_comparison_table_{CACHE_TAG}.parquet"
COMPARISON_CSV = OUTPUT_DIR / f"ims_ol_da_comparison_table_{CACHE_TAG}.csv"
PAIR_DAILY_PARQUET = OUTPUT_DIR / f"ims_ol_da_pair_daily_{CACHE_TAG}.parquet"
PAIR_DAILY_CSV = OUTPUT_DIR / f"ims_ol_da_pair_daily_{CACHE_TAG}.csv"
SCOPE_META_CSV = OUTPUT_DIR / f"ims_ol_da_scope_metadata_{CACHE_TAG}.csv"
CELL_COUNTS_NC = OUTPUT_DIR / f"ims_ol_da_cell_counts_metrics_{CACHE_TAG}.nc4"

print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"COMPARISON={COMPARISON_PARQUET if COMPARISON_PARQUET.exists() else COMPARISON_CSV}")
print(f"PAIR_DAILY={PAIR_DAILY_PARQUET if PAIR_DAILY_PARQUET.exists() else PAIR_DAILY_CSV}")
print(f"SCOPE_META={SCOPE_META_CSV}")
print(f"CELL_COUNTS_NC={CELL_COUNTS_NC}")


In [ ]:
if COMPARISON_PARQUET.exists():
    comparison_df = pd.read_parquet(COMPARISON_PARQUET)
else:
    comparison_df = pd.read_csv(COMPARISON_CSV)

if PAIR_DAILY_PARQUET.exists():
    pair_daily = pd.read_parquet(PAIR_DAILY_PARQUET)
else:
    pair_daily = pd.read_csv(PAIR_DAILY_CSV)

scope_meta = pd.read_csv(SCOPE_META_CSV)

print("comparison_df:", comparison_df.shape)
print("pair_daily:", pair_daily.shape)
print("scope_meta:", scope_meta.shape)

all_period_tbl = comparison_df[comparison_df["scope"] == "ALL_PERIOD"].copy()
all_period_tbl = all_period_tbl.set_index("metric").reindex(["accuracy", "hit_rate", "miss_rate", "false_alarm_ratio", "correct_rejection_rate"]).reset_index()
all_period_tbl


In [ ]:
season_tbl = comparison_df[comparison_df["scope"] == "SEASON_ALL_YEARS"].copy()
if not season_tbl.empty:
    season_tbl["season"] = pd.Categorical(season_tbl["season"], categories=SEASON_ORDER, ordered=True)
season_tbl = season_tbl.sort_values(["season", "metric"]).reset_index(drop=True)
season_tbl


In [ ]:
ds = xr.open_dataset(CELL_COUNTS_NC)

print(ds)

ny = int(ds.attrs["grid_ny"])
nx = int(ds.attrs["grid_nx"])
cell_i = ds["cell_i"].values.astype(np.int64)
cell_j = ds["cell_j"].values.astype(np.int64)

EXP_INDEX = {"OL": 0, "DA": 1}
METRICS = ["accuracy", "hit_rate", "miss_rate", "false_alarm_ratio", "correct_rejection_rate"]

scope_df = scope_meta.copy()
scope_df


In [ ]:
def find_scope_id(scope: str, year=None, season=None) -> int:
    sub = scope_df[scope_df["scope"] == scope].copy()
    if year is None:
        sub = sub[sub["year"] == -1]
    else:
        sub = sub[sub["year"] == int(year)]
    if season is None:
        sub = sub[sub["season"] == "ALL"]
    else:
        sub = sub[sub["season"] == str(season)]
    if sub.empty:
        raise KeyError(f"No scope row found for scope={scope}, year={year}, season={season}")
    return int(sub.iloc[0]["scope_id"])


def values_to_grid(values_1d: np.ndarray) -> np.ndarray:
    g = np.full((ny, nx), np.nan, dtype=np.float32)
    g[cell_j, cell_i] = np.asarray(values_1d, dtype=np.float32)
    return g


def get_metric_grid(metric: str, experiment: str, scope: str, year=None, season=None) -> np.ndarray:
    exp_i = EXP_INDEX[experiment]
    sid = find_scope_id(scope, year=year, season=season)
    vals = ds[metric].isel(experiment=exp_i, scope=sid).values
    return values_to_grid(vals)


def get_delta_grid(metric: str, scope: str, year=None, season=None) -> np.ndarray:
    g_da = get_metric_grid(metric, "DA", scope, year=year, season=season)
    g_ol = get_metric_grid(metric, "OL", scope, year=year, season=season)
    return g_da - g_ol


def plot_grid(grid: np.ndarray, title: str, cmap="viridis", vmin=None, vmax=None):
    plt.figure(figsize=(11, 5))
    plt.imshow(grid, origin="upper", cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar(shrink=0.8)
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
metric = "accuracy"
scope = "ALL_PERIOD"

g_ol = get_metric_grid(metric, "OL", scope)
g_da = get_metric_grid(metric, "DA", scope)
g_d = get_delta_grid(metric, scope)

plot_grid(g_ol, f"{metric} OL ({scope})", cmap="viridis", vmin=0, vmax=1)
plot_grid(g_da, f"{metric} DA ({scope})", cmap="viridis", vmin=0, vmax=1)
plot_grid(g_d, f"{metric} DA-OL ({scope})", cmap="RdBu_r", vmin=-0.5, vmax=0.5)


In [ ]:
# Example seasonal map
metric = "hit_rate"
scope = "SEASON_ALL_YEARS"
season = "DJF"

g_djf_da = get_metric_grid(metric, "DA", scope, season=season)
g_djf_ol = get_metric_grid(metric, "OL", scope, season=season)
g_djf_delta = get_delta_grid(metric, scope, season=season)

plot_grid(g_djf_ol, f"{metric} OL ({scope}, {season})", cmap="viridis", vmin=0, vmax=1)
plot_grid(g_djf_da, f"{metric} DA ({scope}, {season})", cmap="viridis", vmin=0, vmax=1)
plot_grid(g_djf_delta, f"{metric} DA-OL ({scope}, {season})", cmap="RdBu_r", vmin=-0.5, vmax=0.5)
